# Data Cleaning and Preprocessing

This notebook accompanies the QSAR tutorial chapter: **Data Cleaning and Preprocessing**.

## 1. Removing invalid SMILES

In [ ]:
from pathlib import Path
import pandas as pd
from rdkit import Chem

def find_example_dataset():
    candidates = [
        Path("../data/example_qsar_dataset.csv"),
        Path("data/example_qsar_dataset.csv"),
        Path("../../data/example_qsar_dataset.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not find example_qsar_dataset.csv.")

df = pd.read_csv(find_example_dataset())

def smiles_to_mol(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

df["Mol"] = df["SMILES"].apply(smiles_to_mol)

num_invalid = df["Mol"].isna().sum()
print("Original dataset size:", df.shape)
print("Number of invalid SMILES:", num_invalid)

invalid_rows = df[df["Mol"].isna()]
display(invalid_rows[["CompoundID", "Name", "SMILES"]])

df_clean = df.dropna(subset=["Mol"]).copy()
print("Dataset size after removing invalid SMILES:", df_clean.shape)

## Calculate descriptors for cleaned molecules

In [ ]:
import numpy as np
from rdkit.Chem import Descriptors

descriptor_rows = [
    Descriptors.CalcMolDescriptors(mol, missingVal=np.nan)
    for mol in df_clean["Mol"]
]

X = pd.DataFrame(descriptor_rows, index=df_clean.index)
X = X.apply(pd.to_numeric, errors="coerce").dropna(axis=1)

y = df_clean["Activity"].astype(float)

print("Descriptor matrix shape:", X.shape)
display(X.head())

## 2. Removing constant and near-constant descriptors

In [ ]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=1e-8)

X_variance_filtered_array = selector.fit_transform(X)
selected_columns = X.columns[selector.get_support()]

X_variance_filtered = pd.DataFrame(
    X_variance_filtered_array,
    columns=selected_columns,
    index=X.index
)

print("Original number of descriptors:", X.shape[1])
print("After variance filtering:", X_variance_filtered.shape[1])

## Important: split before correlation filtering

Highly correlated descriptors should be identified using the **training set only**.
Then remove those same descriptor columns from the test set.

Do **not** calculate descriptor correlation using the combined train+test dataset,
because that leaks information from the test set into preprocessing decisions.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_variance_filtered,
    y,
    test_size=0.2,
    random_state=42
)

def find_highly_correlated_features(X_train, threshold=0.95):
    corr_matrix = X_train.corr().abs()

    upper_triangle = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    columns_to_drop = [
        column for column in upper_triangle.columns
        if any(upper_triangle[column] > threshold)
    ]

    return columns_to_drop

columns_to_drop = find_highly_correlated_features(X_train, threshold=0.95)

X_train_filtered = X_train.drop(columns=columns_to_drop)
X_test_filtered = X_test.drop(columns=columns_to_drop)

print("Descriptors before correlation filtering:", X_train.shape[1])
print("Descriptors after correlation filtering:", X_train_filtered.shape[1])
print("Descriptors removed:", len(columns_to_drop))
print(columns_to_drop[:10])

## 3A. Random splitting

In [ ]:
from sklearn.model_selection import train_test_split

X_random_train, X_random_test, y_random_train, y_random_test = train_test_split(
    X_variance_filtered,
    y,
    test_size=0.2,
    random_state=42
)

print("Random training set size:", X_random_train.shape[0])
print("Random test set size:", X_random_test.shape[0])
print("Training target range:", y_random_train.min(), "to", y_random_train.max())
print("Test target range:", y_random_test.min(), "to", y_random_test.max())

## 3B. Sorted split based on the target variable

In [ ]:
import numpy as np

data_for_split = X_variance_filtered.copy()
data_for_split["target"] = y

data_for_split = data_for_split.sort_values(by="target").reset_index(drop=True)

test_mask = np.arange(len(data_for_split)) % 5 == 0

test_data = data_for_split[test_mask]
train_data = data_for_split[~test_mask]

X_sorted_train = train_data.drop(columns=["target"])
y_sorted_train = train_data["target"]

X_sorted_test = test_data.drop(columns=["target"])
y_sorted_test = test_data["target"]

print("Sorted training set size:", X_sorted_train.shape[0])
print("Sorted test set size:", X_sorted_test.shape[0])
print("Training target range:", y_sorted_train.min(), "to", y_sorted_train.max())
print("Test target range:", y_sorted_test.min(), "to", y_sorted_test.max())

## 3C. Cluster splitting with UMAP

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

X_scaled = StandardScaler().fit_transform(X_variance_filtered)

try:
    import umap.umap_ as umap

    reducer = umap.UMAP(
        n_components=2,
        random_state=42
    )
    X_embedded = reducer.fit_transform(X_scaled)
    method_name = "UMAP"

except ModuleNotFoundError:
    from sklearn.decomposition import PCA

    print("umap-learn is not installed. Install with: pip install umap-learn")
    print("Using PCA fallback so the notebook remains runnable.")
    reducer = PCA(n_components=2, random_state=42)
    X_embedded = reducer.fit_transform(X_scaled)
    method_name = "PCA fallback"

n_clusters = 5

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_embedded)

split_df = pd.DataFrame({
    "index": X_variance_filtered.index,
    "cluster": cluster_labels,
    "target": y.values
})

train_indices = []
test_indices = []

for cluster_id in sorted(split_df["cluster"].unique()):
    cluster_data = split_df[split_df["cluster"] == cluster_id]

    if len(cluster_data) < 3:
        train_indices.extend(cluster_data["index"].tolist())
        continue

    cluster_train, cluster_test = train_test_split(
        cluster_data,
        test_size=0.2,
        random_state=42
    )

    train_indices.extend(cluster_train["index"].tolist())
    test_indices.extend(cluster_test["index"].tolist())

X_cluster_train = X_variance_filtered.loc[train_indices]
y_cluster_train = y.loc[train_indices]

X_cluster_test = X_variance_filtered.loc[test_indices]
y_cluster_test = y.loc[test_indices]

print("Cluster training set size:", X_cluster_train.shape[0])
print("Cluster test set size:", X_cluster_test.shape[0])

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(
    X_embedded[:, 0],
    X_embedded[:, 1],
    c=cluster_labels,
    s=40
)

plt.xlabel(method_name + " 1")
plt.ylabel(method_name + " 2")
plt.title(method_name + " Projection of Molecular Descriptor Space")
plt.show()